In [1]:
import os
import re
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import json
import torch
from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

In [2]:
train_ds = load_from_disk("/kaggle/input/datasets/shubhamnegi1247/test-train-split-for-reduced-features/data/train_dataset")
val_ds   = load_from_disk("/kaggle/input/datasets/shubhamnegi1247/test-train-split-for-reduced-features/data/val_dataset")

print(train_ds, val_ds)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
    num_rows: 2251026
}) Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
    num_rows: 281381
})


In [3]:
num_workers = max(2, os.cpu_count() // 2)
with open("/kaggle/input/datasets/shubhamnegi1247/test-train-split-for-reduced-features/data/class_weights_balanced.json") as f:
    class_weights = json.load(f)

In [4]:
labels = sorted(class_weights.keys())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

label_names = labels

In [5]:
weights_tensor = torch.tensor(
    [class_weights[id2label[i]] for i in range(len(id2label))],
    dtype=torch.float
)

In [6]:
MODEL_NAME = "markusbayer/CySecBERT" # replace with CysecBERT if needed

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label
)

config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: markusbayer/CySecBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) # not needed if already tokenized

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [8]:
def _safe_metric_name(name: str) -> str:
    # sanitize label names for metric keys
    return re.sub(r"[^a-zA-Z0-9_]+", "_", name).strip("_").lower()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    # Per-class recall
    _, per_class_recall, _, _ = precision_recall_fscore_support(
        labels,
        preds,
        labels=list(range(len(label_names))),
        average=None,
        zero_division=0
    )

    metrics = {
        "accuracy": acc,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "macro_f1": f1_macro,
        "weighted_precision": precision_weighted,
        "weighted_recall": recall_weighted,
        "weighted_f1": f1_weighted,
    }

    # Add per-class recall metrics
    for i, rec in enumerate(per_class_recall):
        safe_name = _safe_metric_name(id2label[i])
        metrics[f"recall_{safe_name}"] = rec

    return metrics

In [9]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=weights_tensor.to(logits.device)
        )

        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [10]:
training_args = TrainingArguments(
    output_dir="./checkpoints",

    # ---- Monitoring / checkpointing ----
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    save_total_limit=2,
    
    logging_strategy="steps",
    logging_steps=100,
    logging_first_step=True,
    log_level="info",
    disable_tqdm=True,


    # ---- Batch / throughput ----
    per_device_train_batch_size=16,   # try this first
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,    # global batch = 16 * 2 * 1 = 32

    # ---- Memory / speed ----
    gradient_checkpointing=False,     # faster; use True only if you hit OOM
    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    # ---- Optimizer / schedule ----
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,

    # ---- Training horizon ----
    max_steps=15000,                  # same effective batch as your stable run
    fp16=True,

    # ---- Best model selection ----
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,

    # ---- Misc ----
    remove_unused_columns=True,
    seed=42,
    data_seed=42,
    ddp_find_unused_parameters=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

max_steps is given, it will override any value given in num_train_epochs


In [12]:
def on_the_fly_transform(batch):
    # This runs only on the small batches requested during training
    batch["labels"] = [label2id[label] for label in batch["label"]]
    # Remove the original string label so the collator doesn't try to tensorize it!
    if "label" in batch:
        del batch["label"]
        
    return batch

# Apply transform (uses NO disk space)
train_ds.set_transform(on_the_fly_transform)
val_ds.set_transform(on_the_fly_transform)

In [13]:
# Create a tiny dataloader with padding
sanity_loader = DataLoader(
    train_ds.select(range(8)),
    batch_size=8,
    collate_fn=data_collator
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device (if not already)
model = model.to(device)

# Move batch to same device
batch = next(iter(sanity_loader))
batch = {k: v.to(device) for k, v in batch.items()}  # ← this is what you're missing

out = model(**batch)
print("Logits shape:", out.logits.shape)

Logits shape: torch.Size([8, 15])


In [14]:
trainer.train(resume_from_checkpoint="/kaggle/input/datasets/shubhamnegi1247/training-2-output/checkpoints/checkpoint-8000")

Loading model from /kaggle/input/datasets/shubhamnegi1247/training-2-output/checkpoints/checkpoint-8000.
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.L

{'loss': '0.0554', 'grad_norm': '0.002569', 'learning_rate': '9.789e-06', 'epoch': '0.1151'}
{'loss': '0.05437', 'grad_norm': '0.02562', 'learning_rate': '9.647e-06', 'epoch': '0.1166'}
{'loss': '0.0543', 'grad_norm': '0.006161', 'learning_rate': '9.505e-06', 'epoch': '0.118'}
{'loss': '0.04195', 'grad_norm': '0.004135', 'learning_rate': '9.363e-06', 'epoch': '0.1194'}
{'loss': '0.04789', 'grad_norm': '0.04813', 'learning_rate': '9.221e-06', 'epoch': '0.1208'}
{'loss': '0.04459', 'grad_norm': '0.007307', 'learning_rate': '9.079e-06', 'epoch': '0.1223'}
{'loss': '0.01326', 'grad_norm': '0.01767', 'learning_rate': '8.938e-06', 'epoch': '0.1237'}
{'loss': '0.03575', 'grad_norm': '0.009833', 'learning_rate': '8.796e-06', 'epoch': '0.1251'}
{'loss': '0.05409', 'grad_norm': '19.86', 'learning_rate': '8.654e-06', 'epoch': '0.1265'}



***** Running Evaluation *****
  Num examples = 281381
  Batch size = 64


{'loss': '0.08248', 'grad_norm': '0.007868', 'learning_rate': '8.512e-06', 'epoch': '0.1279'}


Saving model checkpoint to ./checkpoints/checkpoint-9000
Configuration saved in ./checkpoints/checkpoint-9000/config.json


...

{'eval_loss': '0.04356', 'eval_accuracy': '0.9989', 'eval_macro_precision': '0.6891', 'eval_macro_recall': '0.7279', 'eval_macro_f1': '0.7054', 'eval_weighted_precision': '0.9988', 'eval_weighted_recall': '0.9989', 'eval_weighted_f1': '0.9988', 'eval_recall_benign': '0.9994', 'eval_recall_bot': '0.9645', 'eval_recall_ddos': '0.9978', 'eval_recall_dos_goldeneye': '0.999', 'eval_recall_dos_hulk': '0.9984', 'eval_recall_dos_slowhttptest': '0.9891', 'eval_recall_dos_slowloris': '0.9948', 'eval_recall_ftp_patator': '0.9975', 'eval_recall_heartbleed': '0', 'eval_recall_infiltration': '0', 'eval_recall_portscan': '0.9986', 'eval_recall_ssh_patator': '0.9932', 'eval_recall_web_attack_brute_force': '0.9868', 'eval_recall_web_attack_sql_injection': '0', 'eval_recall_web_attack_xss': '0', 'eval_runtime': '4078', 'eval_samples_per_second': '69', 'eval_steps_per_second': '1.078', 'epoch': '0.2132'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./checkpoints/checkpoint-15000/model.safetensors
Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
tokenizer config file saved in ./checkpoints/checkpoint-15000/tokenizer_config.json
Deleting older checkpoint [checkpoints/checkpoint-13000] due to args.save_total_limit


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from ./checkpoints/checkpoint-8000 (score: 0.707880349734891).
Could not locate the best model at ./checkpoints/checkpoint-8000/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


{'train_runtime': '3.844e+04', 'train_samples_per_second': '12.49', 'train_steps_per_second': '0.39', 'train_loss': '0.01978', 'epoch': '0.2132'}


TrainOutput(global_step=15000, training_loss=0.0197845245556285, metrics={'train_runtime': 38441.3712, 'train_samples_per_second': 12.487, 'train_steps_per_second': 0.39, 'train_loss': 0.0197845245556285, 'epoch': 0.21323477148340322})

In [15]:
trainer.save_model("/kaggle/working/final_cysecbert_model_v2")
tokenizer.save_pretrained("/kaggle/working/final_cysecbert_model_v2")

Saving model checkpoint to /kaggle/working/final_cysecbert_model_v2
Configuration saved in /kaggle/working/final_cysecbert_model_v2/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in /kaggle/working/final_cysecbert_model_v2/model.safetensors
Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
tokenizer config file saved in /kaggle/working/final_cysecbert_model_v2/tokenizer_config.json
tokenizer config file saved in /kaggle/working/final_cysecbert_model_v2/tokenizer_config.json


('/kaggle/working/final_cysecbert_model_v2/tokenizer_config.json',
 '/kaggle/working/final_cysecbert_model_v2/tokenizer.json')